# 4.0 Forecast updater
Fits logistic / Gompertz / Avrami curves to the coverage table and tests the forecast. Code from growth-models.md section 7, Part C. Run notebook 3.0 first (it writes data/interim/coverage_auto.csv).

### Cell 1. Load and define the three curve shapes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from mycodemo.config import INTERIM_DATA_DIR

cov = pd.read_csv(INTERIM_DATA_DIR / "coverage_auto.csv")

def logistic(t, k, t50):
    return 1 / (1 + np.exp(-k * (t - t50)))

def gompertz(t, mu, lag):          # A fixed at 1 (100%)
    return np.exp(-np.exp(mu * np.e * (lag - t) + 1))

def avrami(t, k, n):
    return 1 - np.exp(-(k * np.clip(t, 0, None)) ** n)

MODELS = {"logistic": (logistic, [1.0, 5.0]),
          "gompertz": (gompertz, [0.2, 2.0]),
          "avrami":   (avrami,   [0.15, 2.0])}

### Cell 2. Fit one dish and plot** (the "hello world")

In [ ]:
g = cov[cov.series_id == "d005-H"].sort_values("day")
t, c = g["day"].values, g["coverage"].values
tt = np.linspace(0, t.max() + 3, 200)
plt.plot(t, c, "ko", label="photos")
for name, (f, p0) in MODELS.items():
    params, _ = curve_fit(f, t, c, p0=p0, maxfev=10000)
    plt.plot(tt, f(tt, *params), label=name)
plt.xlabel("day"); plt.ylabel("coverage"); plt.legend(); plt.show()

### Cell 3. Day of 90% / 95% from a fitted curve

In [ ]:
def day_reaching(f, params, p, t_max=60):
    tt = np.linspace(0, t_max, 6001)
    hit = np.where(f(tt, *params) >= p)[0]
    return tt[hit[0]] if len(hit) else np.nan

### Cell 4. Which shape fits best? (step 3 of the plan)

In [ ]:
results = []
for sid, g in cov.groupby("series_id"):
    t, c = g["day"].values, g["coverage"].values
    for name, (f, p0) in MODELS.items():
        try:
            params, _ = curve_fit(f, t, c, p0=p0, maxfev=10000)
            rmse = np.sqrt(np.mean((f(t, *params) - c) ** 2))
        except RuntimeError:
            rmse = np.nan
        results.append({"series_id": sid, "model": name, "rmse": rmse})
fit = pd.DataFrame(results).pivot(index="series_id", columns="model", values="rmse")
fit

### Cell 5. The forecasting test (step 4 of the plan)

In [ ]:
rows = []
for sid, g in cov.groupby("series_id"):
    g = g.sort_values("day")
    t, c = g["day"].values, g["coverage"].values
    # "truth": day 90% from the full series
    p_full, _ = curve_fit(logistic, t, c, p0=[1, 5], maxfev=10000)
    true90 = day_reaching(logistic, p_full, 0.90)
    # baseline: average 90% day of the OTHER dishes (filled in below)
    for n_photos in range(3, len(t)):
        try:
            p_part, _ = curve_fit(logistic, t[:n_photos], c[:n_photos], p0=[1, 5], maxfev=10000)
            pred = day_reaching(logistic, p_part, 0.90)
        except RuntimeError:
            pred = np.nan
        rows.append({"series_id": sid, "n_photos": n_photos, "pred90": pred, "true90": true90})
fc = pd.DataFrame(rows)
true_by_dish = fc.groupby("series_id")["true90"].first()
fc["baseline90"] = [true_by_dish.drop(s).mean() for s in fc["series_id"]]
fc["err_model"] = (fc["pred90"] - fc["true90"]).abs()
fc["err_baseline"] = (fc["baseline90"] - fc["true90"]).abs()
summary = fc.groupby("n_photos")[["err_model", "err_baseline"]].median()
summary.plot(marker="o"); plt.ylabel("error in days (median)"); plt.show()
summary